# KYC Risk Classification using Bert Model
This notebook fine-tunes a LLaMA model on KYC data and uploads it to Hugging Face Hub. You can then use the model for classification tasks in workflows like Airflow.

In [ ]:
!pip cache purge


Files removed: 90


In [ ]:
# Install necessary libraries
!pip install transformers datasets scikit-learn torch huggingface-hub beautifulsoup4 requests

## Step 1: Load the Dataset and Prepare Training Data

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from datasets import Dataset
import pandas as pd

import os
os.environ["WANDB_DISABLED"] = "true"

hf_token = "HF_TOKEN_KYC"

# Load and prepare dataset
data = {
    "text": [
        "ABC Corp is a business in the healthcare sector. No issues detected.",
        "XYZ Inc operates in the cannabis sector with past lawsuits.",
        "DEF Corp in the insurance sector, low income, low credit score."
    ],
    "label": ["Approved", "Rejected", "Review Required"]
}
df = pd.DataFrame(data)

# Encode labels
label_map = {'Approved': 0, 'Review Required': 1, 'Rejected': 2}
df['label'] = df['label'].map(label_map)

# Split data
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(), df['label'].tolist(), test_size=0.2, random_state=42
)

# Tokenize input using a publicly available model (e.g., BERT)
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=512)

# Convert to datasets
train_dataset = Dataset.from_dict({'input_ids': train_encodings['input_ids'], 'labels': train_labels})
val_dataset = Dataset.from_dict({'input_ids': val_encodings['input_ids'], 'labels': val_labels})

# Load BERT model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)

# Training setup
training_args = TrainingArguments(
    output_dir="./bert_kyc_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# Train the model
trainer.train()

# Save the model locally
model.save_pretrained('./bert_kyc_model')
tokenizer.save_pretrained('./bert_kyc_model')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


Epoch,Training Loss,Validation Loss
1,No log,1.748877
2,No log,1.785038
3,No log,1.798303


('./bert_kyc_model/tokenizer_config.json',
 './bert_kyc_model/special_tokens_map.json',
 './bert_kyc_model/vocab.txt',
 './bert_kyc_model/added_tokens.json',
 './bert_kyc_model/tokenizer.json')

In [ ]:
pip install huggingface_hub

In [ ]:
!huggingface-cli login



    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) Y
Token is valid (permission: fineGrained).
The token `hf_token_llm` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenti

In [ ]:
!huggingface-cli whoami

ajeshmahto


In [ ]:
import os
from huggingface_hub import HfApi, upload_folder

# Upload to Hugging Face
api = HfApi()
#api.create_repo(repo_id="ajeshmahto/bert-kyc-classifier", private=False)
upload_folder(repo_id="ajeshmahto/bert-kyc-classifier", folder_path="./bert_kyc_model")

print("Model uploaded successfully!")

optimizer.pt:   0%|          | 0.00/876M [00:00<?, ?B/s]

Model uploaded successfully!


In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# TASK 1: Simulate file sensor by loading CSV manually
kyc_file = './sample_kyc_1_rows.csv'
kyc_data = pd.read_csv(kyc_file)

# TASK 2: Web scraping task
def scrape_websites():
    def scrape_website(url):
        try:
            response = requests.get(url, timeout=5)
            soup = BeautifulSoup(response.text, 'html.parser')
            text = soup.get_text().lower()
            print(text)
            keywords = ['lawsuit', 'bad debt', 'marijuana', 'illegal']
            return ', '.join([kw for kw in keywords if kw in text]) or 'No issues'
        except Exception as e:
            return f"Error: {e}"

    kyc_data['website_risk'] = kyc_data['Website URL'].apply(scrape_website)
    return kyc_data

kyc_data = scrape_websites()
print(kyc_data)

# TASK 3: Classification using pre-trained BERT model
def classify_kyc_data(kyc_data):
    model = AutoModelForSequenceClassification.from_pretrained("ajeshmahto/bert-kyc-classifier")
    tokenizer = AutoTokenizer.from_pretrained("ajeshmahto/bert-kyc-classifier")

    kyc_data['input_text'] = kyc_data['Summary'] + " " + kyc_data['website_risk']
    inputs = tokenizer(list(kyc_data['input_text']), padding=True, truncation=True, return_tensors="pt", max_length=512)
    print(inputs)
    outputs = model(**inputs)
    print(outputs)
    predictions = torch.argmax(outputs.logits, axis=1).tolist()
    print(predictions)

    label_map = {0: 'Approved', 1: 'Review Required', 2: 'Rejected'}
    kyc_data['decision'] = [label_map[pred] for pred in predictions]
    return kyc_data

kyc_data = classify_kyc_data(kyc_data)
print(kyc_data)

# TASK 4: Simulate Slack notification
def notify_slack():
    approved = len(kyc_data[kyc_data['decision'] == 'Approved'])
    review = len(kyc_data[kyc_data['decision'] == 'Review Required'])
    rejected = len(kyc_data[kyc_data['decision'] == 'Rejected'])

    message = f"""
    *KYC Underwriting Summary:*
    ✅ Approved: {approved}
    🔍 Review Required: {review}
    ❌ Rejected: {rejected}
    📁 Results saved locally.
    """
    print(message)

notify_slack()


918 family wellness pllc - primary care, weight loss


























 semaglutide starting at just $200 and tirzepatide starting at just $275homeabout usmeet the providerservicesweightloss patient formsinsurance contact uspatient portalschedule an appointment  we accept most major insurance plans                                    welcome  918 family wellness is a locally owned family wellness clinic offering primary care for all ages, weightloss, and iv therapy. we are conveniently located in the heart of owasso at 8430 n 123rd east ave owasso, ok, 74055phone: 918-401-4770fax: 918-401-4779office hours: monday through thursday 8:00am - 5:00pm cst. ***our office is closed 12:00-1:00pm daily for lunch***we accept most insurance plans and private pay options!most insurance is accepted: bcbs, united healthcare, healthchoice, umr,  soonercare plans including aetna better health of ok, humana healthy horizins, ok complete health, medicare, ambetter, aetna and much more! *call the c